# Smart Fitness Recommendation System — Data Preparation & Feature Engineering


# 1. Project introduction

Modern fitness trackers collect detailed workout data such as heart rate, duration, and calories burned. However, many users find it difficult to interpret these numbers and turn them into practical decisions about training and nutrition.

This project develops a Smart Fitness Recommendation System that uses machine learning and simple rules to:

* Predict approximate calories burned from workout and body-related data
* Classify users into broad fitness profiles based on workout intensity
* Recommend suitable healthy meals after exercise

The goal is to transform raw fitness data into understandable, personalised guidance, rather than just displaying numbers. The final system is designed as a prototype that could be integrated into a fitness app or web-based tool.

# 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


# 3. Import dataset

In [ ]:
# Folder path
DATA_DIR = Path(r"C:\Users\hoang\158739 Data Wrangling\Group project\Fitness data")
fitness_df = pd.read_csv(DATA_DIR / "Fitness Tracker Dataset.csv")

print("Shape:", fitness_df.shape)

fitness_df.head()


# 4. Initial Data Understanding

In [ ]:
fitness_df = fitness_df.rename(columns={
    "Weight (kg)": "Weight_kg",
    "Height (m)": "Height_m",
    "Session_Duration (hours)": "Session_Duration_hours",
    "Water_Intake (liters)": "Water_Intake_liters",
    "Workout_Frequency (days/week)": "Workout_Frequency_days"
})

print(fitness_df.columns.tolist())


In [ ]:
for df in [fitness_df]:
    df["Gender"] = df["Gender"].astype(str).str.strip()
    df["Workout_Type"] = df["Workout_Type"].astype(str).str.strip()
    print(df["Gender"].value_counts())
    print(df["Workout_Type"].value_counts())


In [ ]:
# Convert numeric columns
numeric_columns = [
    "Age",
    "Weight_kg",
    "Height_m",
    "Max_BPM",
    "Avg_BPM",
    "Resting_BPM",
    "Session_Duration_hours",
    "Calories_Burned",
    "Fat_Percentage",
    "Water_Intake_liters",
    "Workout_Frequency_days",
    "Experience_Level",
    "BMI"
]

for col in numeric_columns:
    if col in fitness_df.columns:
        fitness_df[col] = pd.to_numeric(fitness_df[col], errors="coerce")


# 5. Data Cleaning

In [ ]:
print("Columns in dataset:")
print(fitness_df.columns.tolist())
print("\nDataset Information:")
fitness_df.info()

print("\nMissing Values:")
print(fitness_df.isnull().sum())

print("\nDuplicated Rows:")
print(fitness_df.duplicated().sum())


In [ ]:
#Fix duplicate rows 
print("Duplicated rows before cleaning:", fitness_df.duplicated().sum())

fitness_df = fitness_df.drop_duplicates().copy()

print("Duplicated rows after cleaning:", fitness_df.duplicated().sum())
display(fitness_df.head())


In [ ]:
# Handle missing values
numeric_columns = fitness_df.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = fitness_df.select_dtypes(include=["object"]).columns

for col in numeric_columns:
    fitness_df[col] = fitness_df[col].fillna(fitness_df[col].median())

for col in categorical_columns:
    fitness_df[col] = fitness_df[col].fillna(fitness_df[col].mode()[0])


# 6. Feature Engineering

In [ ]:
# Correct BMI
fitness_df["BMI_Corrected"] = (
    fitness_df["Weight_kg"] / (fitness_df["Height_m"] ** 2)
)

fitness_df["BMI_Corrected"] = fitness_df["BMI_Corrected"].round(2)
fitness_df.head()


* The original BMI column was checked and replaced with a corrected BMI calculation based on weight and height
* BMI\_Corrected is useful because it gives a more reliable body composition indicator than the original BMI column

In [ ]:
# BMI Category
def get_bmi_category(bmi):
    if pd.isna(bmi):
        return np.nan
    elif bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obese"
        
fitness_df["BMI_Category"] = fitness_df["BMI_Corrected"].apply(get_bmi_category)


* BMI\_Category converts numerical BMI into understandable groups such as underweight, normal, overweight, and obese

In [ ]:
# Session duration in minutes
fitness_df["Session_Minutes"] = (
    fitness_df["Session_Duration_hours"] * 60
)

fitness_df["Session_Minutes"] = fitness_df["Session_Minutes"].round(2)

# Weekly workout minutes
fitness_df["Weekly_Workout_Minutes"] = (
    fitness_df["Session_Minutes"] * fitness_df["Workout_Frequency_days"]
)

fitness_df["Weekly_Workout_Minutes"] = fitness_df["Weekly_Workout_Minutes"].round(2)

# Activity level
def get_activity_level(weekly_minutes):
    if pd.isna(weekly_minutes):
        return np.nan
    elif weekly_minutes < 150:
        return "Low"
    elif weekly_minutes < 300:
        return "Moderate"
    else:
        return "High"

fitness_df["Activity_Level"] = (
    fitness_df["Weekly_Workout_Minutes"].apply(get_activity_level)
)
# Preview results
fitness_df[
    [
        "Session_Minutes",
        "Workout_Frequency_days",
        "Weekly_Workout_Minutes",
        "Activity_Level"
    ]
].head()


* Session\_Minutes - A longer session usually gives the body more time to burn calories
* Weekly\_Workout\_Minutes shows the total amount of exercise a user performs in one week
* Activity\_Level makes weekly workout behaviour easier to understand by grouping users into low, moderate, and high activity levels

In [ ]:
# Age group
def get_age_group(age):
    if pd.isna(age):
        return np.nan
    elif age < 25:
        return "Young Adult"
    elif age < 40:
        return "Adult"
    elif age < 55:
        return "Middle Age"
    else:
        return "Older Adult"

fitness_df["Age_Group"] = fitness_df["Age"].apply(get_age_group)
fitness_df[
    [
        "Age",
        "Age_Group",
        "Session_Minutes",
        "Workout_Frequency_days",
        "Weekly_Workout_Minutes",
        "Activity_Level"
    ]
].head()


* Age\_Group converts age into more understandable groups

In [ ]:
# BPM_Increase
fitness_df["BPM_Increase"] = fitness_df["Avg_BPM"] - fitness_df["Resting_BPM"]
met_map = {
    "Yoga": 3.0,
    "Strength": 5.0,
    "Cardio": 7.0,
    "HIIT": 8.5
}

fitness_df["MET"] = fitness_df["Workout_Type"].map(met_map)
fitness_df["MET"] = fitness_df["MET"].fillna(fitness_df["MET"].median())
fitness_df["MET"].isnull().sum()
fitness_df[[ "MET", 
    "BPM_Increase"]
    ].head()


* BPM\_Increase measures how much the user’s heart rate rises during exercise
* MET gives each workout type an estimated intensity value. This helps the model understand that different workouts have different energy demands

In [ ]:
# Fitness score
fitness_df["Fitness_Score"] = (
    fitness_df["Weekly_Workout_Minutes"]
    * fitness_df["Workout_Frequency_days"]
    * fitness_df["Avg_BPM"]
    * fitness_df["Calories_Burned"]
) / (fitness_df["BMI_Corrected"] * 100000)

fitness_df["Fitness_Score"] = fitness_df["Fitness_Score"].replace(
    [np.inf, -np.inf], np.nan
)

fitness_df["Fitness_Score"] = fitness_df["Fitness_Score"].fillna(
    fitness_df["Fitness_Score"].median()
)

# Normalise fitness score
min_score = fitness_df["Fitness_Score"].min()
max_score = fitness_df["Fitness_Score"].max()

fitness_df["Fitness_Score_100"] = (
    (fitness_df["Fitness_Score"] - min_score) /
    (max_score - min_score)
) * 100

fitness_df["Fitness_Score_100"] = fitness_df["Fitness_Score_100"].round(2)
# Preview results
fitness_df[
    [
        "Fitness_Score",
        "Fitness_Score_100"
    ]
].head()


In [ ]:
# Fitness level
def get_fitness_level(score):
    if pd.isna(score):
        return np.nan
    elif score < 34:
        return "Low Fitness"
    elif score < 67:
        return "Moderate Fitness"
    else:
        return "High Fitness"

fitness_df["Fitness_Level"] = (
    fitness_df["Fitness_Score_100"].apply(get_fitness_level)
)

# Final check
print("Final dataset shape:", fitness_df.shape)
print("\nMissing values after cleaning:")
print(fitness_df.isnull().sum())

display(fitness_df.head())


# 7.Exploratory Data Analysis

In [ ]:
#Graph 1:Age distribution 
plt.figure(figsize=(8, 5))

sns.histplot(
    data=fitness_df,
    x="Age",
    bins=20,
    kde=True
)

plt.title("Distribution of User Age")
plt.xlabel("Age")
plt.ylabel("Number of Users")
plt.show()


* The age distribution shows that users are mainly adults, with a high concentration among younger users
* This suggests that the recommendation system is mostly based on adult fitness behaviour. However, the large spike in the youngest group should be checked to make sure it is not caused by missing-value imputation

In [ ]:
#Graph 2: Gender distribution
plt.figure(figsize=(7, 5))

sns.countplot(
    data=fitness_df,
    x="Gender",
    order=fitness_df["Gender"].value_counts().index
)

plt.title("Gender Distribution of Users")
plt.xlabel("Gender")
plt.ylabel("Number of Users")
plt.show()


* The gender distribution is relatively balanced, with slightly more female users than male users
* This improves the representativeness of the dataset and suggests that the recommendation system can be developed for both male and female users

In [ ]:
#Graph 3: Workout type distribution 
plt.figure(figsize=(8, 5))

sns.countplot(
    data=fitness_df,
    x="Workout_Type",
    order=fitness_df["Workout_Type"].value_counts().index
)

plt.title("Distribution of Workout Types")
plt.xlabel("Workout Type")
plt.ylabel("Number of Records")
plt.xticks(rotation=30)
plt.show()


* Strength, cardio, yoga, and HIIT are the main workout types in the dataset. Strength training appears to be the most common workout type
* This feature is important because workout type can influence calories burned, intensity, and final exercise recommendations.

In [ ]:
fitness_df.info()


In [ ]:
#Graph 4: BMI category distribution
plt.figure(figsize=(8, 5))

bmi_order = ["Underweight", "Normal", "Overweight", "Obese"]

sns.countplot(
    data=fitness_df,
    x="BMI_Category",
    order=[cat for cat in bmi_order if cat in fitness_df["BMI_Category"].unique()]
)

plt.title("BMI Category Distribution")
plt.xlabel("BMI Category")
plt.ylabel("Number of Users")
plt.show()


In [ ]:
#Graph 5: Weekly workout minutes by activity level
plt.figure(figsize=(8, 5))

activity_order = ["Low", "Moderate", "High"]

sns.boxplot(
    data=fitness_df,
    x="Activity_Level",
    y="Weekly_Workout_Minutes",
    order=[level for level in activity_order if level in fitness_df["Activity_Level"].unique()]
)

plt.title("Weekly Workout Minutes by Activity Level")
plt.xlabel("Activity Level")
plt.ylabel("Weekly Workout Minutes")
plt.show()


* The graph shows that users classified as high activity have higher weekly workout minutes than moderate and low activity users
* This confirms that the engineered Activity\_Level feature works as expected. However, because Activity\_Level was created from weekly workout minutes, this graph should be interpreted as a validation of feature engineering rather than an independent finding.

# Graph 6: Correlation heatmap

In [ ]:
numeric_features = [
    "Age",
    "Weight_kg",
    "Height_m",
    "BMI_Corrected",
    "Max_BPM",
    "Avg_BPM",
    "Resting_BPM",
    "BPM_Increase",
    "Session_Minutes",
    "Calories_Burned",
    "Fat_Percentage",
    "Water_Intake_liters",
    "Workout_Frequency_days",
    "Weekly_Workout_Minutes",
    "Fitness_Score_100"
]

available_numeric_features = [
    col for col in numeric_features if col in fitness_df.columns
]

corr_matrix = fitness_df[available_numeric_features].corr()

plt.figure(figsize=(12, 8))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap of Numerical Features")
plt.show()


* The correlation heatmap shows that Fitness Score has the strongest positive relationship with Weekly Workout Minutes (0.71) and Workout Frequency (0.63). This means users who exercise more often and spend more time working out each week tend to have higher fitness scores

# 8. Model 1: Calories Burned Prediction

## 8.1 Model 1: Calories Burned Prediction

The purpose of this model is to predict the number of calories burned during a workout session based on user physical characteristics, workout behaviour, heart rate information, and engineered fitness features. Since `Calories_Burned` is a continuous numerical variable, this is a regression problem.

Several regression models are tested and compared, including Linear Regression, KNN Regressor, Decision Tree Regressor, and Random Forest Regressor.

In [ ]:
# Target variable
target = "Calories_Burned"

# Improved features for calories burned prediction
features_model1 = [
    "Session_Minutes",
    "Avg_BPM",
    "Resting_BPM",
    "BPM_Increase",
    "MET",
    "Weight_kg",
    "Fat_Percentage",
    "Workout_Type"
]

# Create X and y
X = fitness_df[features_model1]
y = fitness_df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 8.2 Split numerical and categorical columns

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")

print(categorical_features)


## 8.3 Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)


## 8.4 Preprocessing pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


## 8.5 Build regression models

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    "Linear Regression": LinearRegression(),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )
}


## 8.6 Train and evaluate models

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

results = []

trained_models = {}

for model_name, model in models.items():
    
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    # Train model
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    
    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })
    
    trained_models[model_name] = pipeline

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(by="R2 Score", ascending=False)

results_df


## 8.7 Interpretation of evaluation metrics

* The regression models were developed to predict calories burned during a workout session using user characteristics, workout behaviour, heart rate information, and engineered fitness features. Among the tested models, Linear Regression performed the best, with an MAE of approximately 254 calories, RMSE of approximately 317 calories, and an R2 score of 0.0028.
* Although Linear Regression had the best result, the R2 score is very close to zero. This means the model explains almost none of the variation in calories burned. The average prediction error is also relatively high, at around 254 calories. Therefore, this model should not be considered an accurate calorie prediction tool.
* The weak performance suggests that calories burned may depend on additional variables not available in the dataset, such as speed, distance, exact workout intensity, heart-rate zones, muscle mass, and individual metabolism. More complex models such as Random Forest and Decision Tree did not improve the result, which suggests that the dataset itself has limited predictive signal for this target variable.
* However, the modelling process is still useful because it follows a proper machine-learning workflow, including train-test splitting, preprocessing, feature scaling, one-hot encoding, model comparison, and evaluation using MAE, RMSE, and R2. For the final Smart Fitness Recommendation System, this model should be treated as a supporting benchmark, while the main recommendation system should rely more on fitness level, activity level, BMI category, clustering, and rule-based workout recommendations.

# 9. Model 2: Fitness Level Classification

## 9.1 Select features

In [ ]:
#Check target distribution
fitness_df["Fitness_Level"].value_counts()


In [ ]:
fitness_df["Fitness_Level"].value_counts(normalize=True) * 100


In [ ]:
# Target variable for Model 2
target_model2 = "Fitness_Level"

# Features used for fitness level classification
features_model2 = [
    "Age",
    "Gender",
    "Weight_kg",
    "Height_m",
    "BMI_Corrected",
    "BMI_Category",
    "Session_Minutes",
    "Weekly_Workout_Minutes",
    "Activity_Level",
    "Avg_BPM",
    "Resting_BPM",
    "BPM_Increase",
    "MET",
    "Workout_Type",
    "Fat_Percentage",
    "Water_Intake_liters",
    "Workout_Frequency_days",
    "Experience_Level",
    "Age_Group"
]

# Create X and y
X = fitness_df[features_model2]
y = fitness_df[target_model2]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 9.2 Split numerical and categorical features

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


## 9.3 Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)


## 9.4 Preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## 9.5 Build classification models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

classification_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    
    "KNN Classifier": KNeighborsClassifier(
        n_neighbors=7
    ),
    
    "Decision Tree Classifier": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=20,
        random_state=42
    ),
    
    "Random Forest Classifier": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=42
    ),
    
    "Gradient Boosting Classifier": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}


## 9.6 Train models

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

classification_results = []
trained_classification_models = {}

for model_name, model in classification_models.items():
    
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    # Train model
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    
    # Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    
    classification_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Weighted F1": f1,
        "Macro F1": macro_f1
    })
    
    trained_classification_models[model_name] = pipeline

classification_results_df = pd.DataFrame(classification_results)

classification_results_df = classification_results_df.sort_values(
    by="Weighted F1",
    ascending=False
)

classification_results_df


In [ ]:
best_classification_model_name = classification_results_df.iloc[0]["Model"]
best_classification_model = trained_classification_models[best_classification_model_name]
print("Best classification model:", best_classification_model_name)


## 9.7 interpretation

* Gradient Boosting Classifier achieved the best overall performance, with 95% accuracy and a weighted F1-score of 0.946. This indicates that the model can classify most users into the correct engineered fitness level
* However, Fitness\_Level was created from your own formula, it is an engineered label, not a real clinical label
* The Macro F1-score was much lower than the Weighted F1-score. This suggests that the dataset may have class imbalance, where one fitness level has many more records than the others. As a result, the model may perform very well on the majority class but less effectively on smaller fitness-level groups
* Compared with Model 1, this classification model is more suitable for the Smart Fitness Recommendation System because the output can be directly used to support personalised workout recommendations. For example, users classified as Low Fitness can receive beginner-friendly recommendations, while users classified as High Fitness can receive more advanced workout plans.
* It is important to note that Fitness\_Level is an engineered label created from the project’s fitness score. Therefore, the model should not be interpreted as a medical diagnosis. Instead, it should be understood as a project-based classification tool that supports fitness recommendation logic.

## 9.8 confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
y_pred_best = best_classification_model.predict(X_test)

# Classification report
print(classification_report(y_test, y_pred_best, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix - Best Classification Model")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()


* In the test set, there were 341 Low Fitness users, 18 Moderate Fitness users, and only 1 High Fitness user. The model correctly classified 335 out of 341 Low Fitness users, showing very strong performance for this majority class. However, it failed to correctly classify the only High Fitness user and correctly classified only 7 out of 18 Moderate Fitness users
* This explains why the weighted average score is high while the macro average F1-score is much lower. Weighted F1 is strongly influenced by the majority class, while macro F1 treats all classes equally. Therefore, the low macro F1-score indicates that the model does not perform equally well across all fitness levels
* Overall, the model is useful for identifying Low Fitness users, but it is less reliable for distinguishing Moderate Fitness and High Fitness users. This limitation is mainly caused by class imbalance in the engineered Fitness\_Level variable. For future improvement, the fitness-level thresholds should be adjusted or quantile-based grouping should be used to create more balanced classes

# 10. Model 3: Fitness Profile Clustering

## 10.1: Select features

In [ ]:
 #Features selected for clustering
cluster_features = [
    "Age",
    "BMI_Corrected",
    "Fat_Percentage",
    "Session_Minutes",
    "Weekly_Workout_Minutes",
    "BPM_Increase",
    "MET",
    "Water_Intake_liters",
    "Workout_Frequency_days",
    "Experience_Level"
]

X_cluster = fitness_df[cluster_features].copy()

print("Clustering feature shape:", X_cluster.shape)
X_cluster.head()


## 10.2 Scale the clustering data

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_cluster_scaled = scaler.fit_transform(X_cluster)

X_cluster_scaled.shape


## 10.3 Find suitable number of clusters

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


inertia_values = []
silhouette_scores = []

k_values = range(2, 8)

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    cluster_labels = kmeans.fit_predict(X_cluster_scaled)
    
    inertia_values.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster_scaled, cluster_labels))

cluster_eval_df = pd.DataFrame({
    "k": list(k_values),
    "Inertia": inertia_values,
    "Silhouette Score": silhouette_scores
})

cluster_eval_df


* The highest silhouette score is at k = 2, which means two clusters give the best mathematical separation
* However, the score is still low, showing that the users are not clearly separated into very distinct groups
* For this project, k = 3 can still be used because three groups are easier to explain for a fitness recommendation system: beginner, moderate, and advanced

## 10.4 Elbow plot

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    cluster_eval_df["k"],
    cluster_eval_df["Inertia"],
    marker="o"
)

plt.title("Elbow Method for Choosing Number of Clusters")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.show()


* The elbow plot does not show a very clear elbow point. This means there is no obvious best number of clusters from the inertia result

## 10.5 Silhouette score graph

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    cluster_eval_df["k"],
    cluster_eval_df["Silhouette Score"],
    marker="o"
)

plt.title("Silhouette Score for Different k Values")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.show()


* The silhouette graph suggests that k = 2 gives the clearest separation
* However, k = 3 is still acceptable for this project because it creates more useful fitness profile groups for recommendation: low activity, moderate fitness, and high activity

## 10.6 Build final K-Means model

In [ ]:
# Final KMeans model
final_k = 3

kmeans_final = KMeans(
    n_clusters=final_k,
    random_state=42,
    n_init=10
)

fitness_df["Fitness_Profile_Cluster"] = kmeans_final.fit_predict(X_cluster_scaled)

fitness_df["Fitness_Profile_Cluster"].value_counts().sort_index()


## 10.7 Analyse cluster characteristics

In [ ]:
cluster_summary = fitness_df.groupby("Fitness_Profile_Cluster")[cluster_features].mean().round(2)

cluster_summary


In [ ]:
#category distribution
fitness_df.groupby("Fitness_Profile_Cluster")[
    ["BMI_Category", "Activity_Level", "Fitness_Level", "Workout_Type"]
].agg(lambda x: x.value_counts().index[0])


## 10.8 PCA cluster visualisation

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)

X_pca = pca.fit_transform(X_cluster_scaled)

pca_df = pd.DataFrame({
    "PCA1": X_pca[:, 0],
    "PCA2": X_pca[:, 1],
    "Fitness_Profile_Cluster": fitness_df["Fitness_Profile_Cluster"]
})

plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=pca_df,
    x="PCA1",
    y="PCA2",
    hue="Fitness_Profile_Cluster",
    palette="Set2",
    alpha=0.7
)

plt.title("Fitness Profile Clusters Visualised with PCA")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(title="Cluster")
plt.show()


* The PCA visualisation shows that Cluster 0 and Cluster 1 have some separation along the first PCA component, while Cluster 2 overlaps with both groups
* This means the clustering model found some structure in the data, but the user groups are not fully separated
* Therefore, the clusters are useful for general recommendation logic, but they should not be interpreted as perfectly distinct fitness categories

## 10.9 Automatically label clusters

In [ ]:
cluster_summary_for_label = fitness_df.groupby("Fitness_Profile_Cluster")[
    ["Weekly_Workout_Minutes", "Experience_Level", "BPM_Increase", "MET"]
].mean()

cluster_summary_for_label


In [ ]:
# Rank clusters by overall activity level
cluster_activity_rank = (
    cluster_summary_for_label["Weekly_Workout_Minutes"]
    + cluster_summary_for_label["Experience_Level"] * 50
    + cluster_summary_for_label["BPM_Increase"]
).sort_values()

cluster_order = cluster_activity_rank.index.tolist()

cluster_name_map = {
    cluster_order[0]: "Beginner / Low Activity Profile",
    cluster_order[1]: "Moderate Fitness Profile",
    cluster_order[2]: "High Activity / Advanced Profile"
}

fitness_df["Fitness_Profile_Label"] = fitness_df["Fitness_Profile_Cluster"].map(cluster_name_map)

fitness_df[[
    "Fitness_Profile_Cluster",
    "Fitness_Profile_Label"
]].head()


## 10.10: Cluster table

In [ ]:
profile_summary = fitness_df.groupby("Fitness_Profile_Label")[cluster_features].mean().round(2)

profile_summary


## 10.11: Interpretation

* K-Means clustering was used to group users into fitness profiles based on body composition, workout behaviour, heart-rate response, workout intensity, hydration, and experience level.
* The cluster evaluation results show that the silhouette scores are generally low, with the highest score occurring at k = 2. This indicates that the dataset does not contain strongly separated natural clusters. The elbow plot also does not show a very clear elbow point, meaning there is no obvious best number of clusters based only on inertia.
* However, k = 3 was selected because it is more useful for the Smart Fitness Recommendation System. Three clusters allow the users to be interpreted as Beginner / Low Activity Profile, Moderate Fitness Profile, and High Activity / Advanced Profile. These groups are easier to connect with practical workout recommendations.
* The PCA visualisation shows that the clusters overlap, especially around the centre of the plot. This means the clusters should not be treated as strict fitness categories. Instead, they should be interpreted as general user profiles that help support recommendation logic
* The cluster summary table shows that the High Activity / Advanced Profile has the highest weekly workout minutes and session duration, while the Beginner / Low Activity Profile has the lowest activity level. The Moderate Fitness Profile sits between the two groups. Therefore, the clustering model is useful as a supporting tool for grouping users and generating personalised workout recommendations, even though the statistical separation between clusters is weak.

## 10.12: cluster rule-based recommendation

In [ ]:
#simple rule-based recommendation logic system
def recommend_by_cluster(profile_label):
    if profile_label == "Beginner / Low Activity Profile":
        return "Low to moderate intensity: walking, beginner yoga, light cardio, 3 days per week, 30–40 minutes per session."
    
    elif profile_label == "Moderate Fitness Profile":
        return "Moderate intensity: cardio, strength training, cycling, swimming, 4 days per week, 45–60 minutes per session."
    
    else:
        return "Moderate to high intensity: HIIT, running, advanced strength training, 5 days per week, 60–75 minutes per session."

fitness_df["Cluster_Recommendation"] = fitness_df["Fitness_Profile_Label"].apply(recommend_by_cluster)

fitness_df[["Fitness_Profile_Label", "Cluster_Recommendation"]].head()


# 11. Recommendation System Design

This section designs the final Smart Fitness Recommendation System by combining engineered features, fitness level classification, and fitness profile clustering.

The recommendation system uses several important outputs from the previous stages:

* `BMI_Category`: shows the user’s body composition group
* `Activity_Level`: shows the user’s weekly activity level
* `Fitness_Level`: classifies users into Low, Moderate, or High Fitness
* `Fitness_Profile_Label`: groups users into broader fitness profiles using clustering
* `Workout_Type`: shows the user’s current workout preference or activity type

The system then applies rule-based logic to recommend suitable workout intensity, exercise type, workout frequency, session duration, and a short safety note.

This recommendation system is not a medical diagnosis. It provides general fitness guidance based on the dataset and engineered features

## 11.1 Recommendation function

In [ ]:
def generate_fitness_recommendation(row):
    fitness_level = row["Fitness_Level"]
    profile = row["Fitness_Profile_Label"]
    bmi_category = row["BMI_Category"]
    activity_level = row["Activity_Level"]
    age_group = row["Age_Group"]
    workout_type = row["Workout_Type"]
    
    # Default recommendation
    recommended_intensity = "Moderate"
    recommended_workout = "Cardio and strength training"
    suggested_frequency = "4 days per week"
    suggested_duration = "45–60 minutes per session"
    safety_note = "Maintain proper form and increase intensity gradually."
    
    # Low fitness / beginner profile
    if fitness_level == "Low Fitness" or profile == "Beginner / Low Activity Profile":
        recommended_intensity = "Low to Moderate"
        recommended_workout = "Walking, beginner yoga, light cardio, basic bodyweight exercises"
        suggested_frequency = "3 days per week"
        suggested_duration = "30–40 minutes per session"
        safety_note = "Start slowly and focus on consistency before increasing intensity."
    
    # Moderate fitness profile
    elif fitness_level == "Moderate Fitness" or profile == "Moderate Fitness Profile":
        recommended_intensity = "Moderate"
        recommended_workout = "Cardio, strength training, cycling, swimming, yoga"
        suggested_frequency = "4 days per week"
        suggested_duration = "45–60 minutes per session"
        safety_note = "Balance cardio, strength, and recovery for steady improvement."
    
    # High fitness / advanced profile
    elif fitness_level == "High Fitness" or profile == "High Activity / Advanced Profile":
        recommended_intensity = "Moderate to High"
        recommended_workout = "HIIT, running, advanced strength training, circuit training"
        suggested_frequency = "5 days per week"
        suggested_duration = "60–75 minutes per session"
        safety_note = "Use progressive overload and include recovery days to avoid overtraining."
    
    # BMI-based adjustment
    if bmi_category in ["Overweight", "Obese"]:
        recommended_workout = "Low-impact cardio, cycling, swimming, walking, beginner strength training"
        safety_note = "Low-impact exercises are recommended to reduce joint pressure."
    
    elif bmi_category == "Underweight":
        recommended_workout = "Strength training, resistance exercises, yoga, light cardio"
        safety_note = "Focus on strength building and avoid excessive high-intensity cardio."
    
    # Older adult adjustment
    if age_group == "Older Adult":
        recommended_intensity = "Low to Moderate"
        suggested_duration = "30–45 minutes per session"
        safety_note = "Prioritise low-impact exercise, mobility, and safe progression."
    
    return pd.Series({
        "Recommended_Intensity": recommended_intensity,
        "Recommended_Workout": recommended_workout,
        "Suggested_Frequency": suggested_frequency,
        "Suggested_Duration": suggested_duration,
        "Recommendation_Note": safety_note
    })


## 11.2 Apply recommendation system to dataset

In [ ]:
recommendation_results = fitness_df.apply(generate_fitness_recommendation, axis=1)

fitness_df = pd.concat([fitness_df, recommendation_results], axis=1)

fitness_df[
    [
        "Age",
        "BMI_Category",
        "Activity_Level",
        "Fitness_Level",
        "Fitness_Profile_Label",
        "Recommended_Intensity",
        "Recommended_Workout",
        "Suggested_Frequency",
        "Suggested_Duration",
        "Recommendation_Note"
    ]
].head()


## 11.3 Check recommendation output

In [ ]:
fitness_df[
    [
        "Fitness_Level",
        "Fitness_Profile_Label",
        "BMI_Category",
        "Recommended_Intensity",
        "Recommended_Workout",
        "Suggested_Frequency",
        "Suggested_Duration"
    ]
].sample(10, random_state=42)


## 11.4 Recommendation summary by fitness profile

In [ ]:
recommendation_summary = fitness_df.groupby("Fitness_Profile_Label")[
    [
        "Recommended_Intensity",
        "Recommended_Workout",
        "Suggested_Frequency",
        "Suggested_Duration"
    ]
].agg(lambda x: x.value_counts().index[0])

recommendation_summary


* Although some users are grouped into the High Activity / Advanced Profile, the recommendation system still considers their predicted fitness level and BMI category. Therefore, if a user has Low Fitness, Overweight, Obese, or Underweight status, the system gives safer and lower-intensity recommendations. This makes the recommendation more cautious and suitable for general users

## 11.5 Interpretation of Recommendation System Design

* The recommendation system converts the results of feature engineering, classification, and clustering into practical workout guidance. Users are recommended different workout intensity levels, exercise types, frequencies, and session durations based on their BMI category, activity level, fitness level, and cluster-based fitness profile.
* For users in the Beginner / Low Activity Profile, the system recommends low to moderate intensity exercises such as walking, beginner yoga, light cardio, and bodyweight exercises. For users in the Moderate Fitness Profile, the system recommends a balanced plan including cardio, strength training, cycling, swimming, and yoga. For users in the High Activity / Advanced Profile, the system recommends more challenging workouts such as HIIT, running, circuit training, and advanced strength training.
* The system also adjusts recommendations based on BMI category and age group. For example, overweight or obese users are recommended lower-impact exercises to reduce joint pressure, while older adults are recommended lower to moderate intensity workouts with shorter session duration.
* Overall, this recommendation design connects the machine learning results to a practical fitness application. It makes the project more complete because the system does not only analyse data, but also provides user-friendly workout recommendations.

# 12. Streamlit App description

## 12.1: App introduction and inputs

* Smart Fitness Recommendation System, the purpose of the app is to allow users to enter their personal fitness information and receive instant fitness-related outputs, including BMI category, activity level, calorie prediction, fitness level classification, user fitness profile, workout recommendation, and optional diet recommendation
* Apps input: Age, Gender, Height and Weight, Workout Type, Session Duration, Workout Frequency, Average BPM and Resting BPM, Fat Percentage, Water Intake, Experience Level, Fitness Goal
* From these inputs, the app creates additional engineered features such as BMI, BMI category, weekly workout minutes, activity level, age group, and BPM increase

## 12.2: Models using

* Model 1: Calorie Prediction Model: This model predicts the estimated number of calories burned during a workout session. It uses features such as session duration, average BPM, resting BPM, BPM increase, MET value, body weight, fat percentage, and workout type.
* If the saved calorie model cannot be loaded, the app uses a fallback calorie formula: Calories = MET × 3.5 × Weight / 200 × Session Minutes
* Model 2: Fitness Level Classification: This model classifies the user into a fitness level, such as Low Fitness, Moderate Fitness, or High Fitness. It uses demographic information, body measurements, workout habits, heart rate features, and lifestyle-related variables
* Model 3: Fitness Profile Clustering: The third model uses K-Means clustering to group users into fitness profiles. The app uses the saved scaler and K-Means model to assign each user to a cluster. The cluster is then converted into a more understandable label, such as Beginner - Moderate - High Activity

## 12.3 Recommendation Logic

* After generating the model outputs, the app combines the predicted fitness level, cluster profile, BMI category, activity level, age group, and user goal to produce a final workout recommendation
* For example, a beginner user may receive low-impact exercises such as walking, yoga, and light cardio. A more advanced user may receive HIIT, running, strength training, or circuit training. The recommendation also changes based on the user’s goal, such as weight loss, muscle gain, or general fitness

## 12.4: App output

* BMI score
* BMI category
* Activity level
* Predicted calories burned
* Predicted fitness level
* Fitness profile
* MET value
* Workout recommendation
* Recommended intensity
* matching diet or exercise recommendations

# 13. Limitations

* The recommendation system is based on a student project dataset, so the results should be treated as general fitness guidance rather than professional medical advice
* Some features such as average BPM, resting BPM, fat percentage, and water intake, still need to be entered manually by the user. In a real-world app, these values could be collected automatically from fitness trackers or smart watches
* The quality of prediction depends on the quality and size of the training dataset. If the dataset is limited, the model may not generalize well to all types of users

# 14. Conclusion

* Overall, the project achieved its main objective: building a clear and understandable fitness recommendation app based on data science techniques. The strongest part of the project is the combination of machine learning outputs and rule-based recommendation logic, which makes the final recommendation more practical for users. The Streamlit interface is also simple and user-friendly, making it suitable for demonstration and presentation
* However, the project still has some limitations. Some user inputs, such as average BPM, resting BPM, fat percentage, and water intake, must be entered manually. In a real fitness application, these values could be collected automatically from smart watches or fitness trackers. In addition, the accuracy of the models depends on the quality and size of the dataset, so the system should be seen as a student project prototype rather than a professional medical or fitness product, and Model 1 calorise burned prediction result was weak
* In the future, the app could be improved by using a larger dataset, connecting with wearable devices, improving the diet recommendation system, and adding more personalised workout plans. Despite these limitations, the project demonstrates how data preparation, feature engineering, machine learning, and web app development can be combined to solve a real-world fitness recommendation problem